In [ ]:
%pip install kagglehub catboost xgboost tqdm -q
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
tqdm.pandas()
import kagglehub
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
Food_path = os.path.join(path,'Q1_data.csv')
df = pd.read_csv(Food_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()# we have to take off out layers , because of the the pic is bad now

In [ ]:
# Task 1: Write your code here:Drop the 'Order_ID' column from the data
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
df_mode = df.copy()#Fill with mode (categorical)
df_mode['Weather'] = df_mode['Weather'].fillna(df_mode['Weather'].mode()[0])
print("\n4. Fill with mode:")
print(df_mode)

# Strategy 3: Fill with mean/median (numerical)
df_mean = df.copy()
df_mean['Courier_Experience_yrs'] = df_mean['Courier_Experience_yrs'].fillna(df_mean['Courier_Experience_yrs'].median())
print("\n3. Fill with mean/median:")
print(df_mean)

#for over all missing data : Drop rows with ANY missing values
df_dropna = df.dropna()
print("\n1. After dropna():", len(df_dropna), "rows")

# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = ['Delivery_Time','Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
df_clean = df[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Courier_Experience_yrs', 'Weather'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")



In [ ]:
# Task 3: Write your code here:Check and remove duplicates if any exist
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 4: Write your code here:Encode categorical variables if needed (Bonus if used One Hot Encoding)
#Traffic_Level
"""
# Example: Encoding car colors
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

print('data before encoding:\n',df.Traffic_Level) #show before encoding

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(Traffic_Level)

print("Original:", df.Traffic_Level)
print("Encoded:\n", encoded)"""

X = df.drop('Delivery_Time', axis=1)# drop the Target
y = df['Delivery_Time']#save target her
categorical_cols = ['Traffic_Level', 'Weather','Vehicle_Type']
# Encode Categorical Features --> make them numbers
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()


for col in X.select_dtypes(include=["object"]).columns: #for loop on x , if it Obj
    X[col] = label_encoder.fit_transform(X[col])#make them uniqe int num




In [ ]:
# Task 5: Write your code here:Apply feature scaling for all features (Use StandardScaler)
# Feature Scaling
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()# all num in same range
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True)) #Counts how many times each unique value appears in the column.(normalize:returns proportions (fractions))
  sns.countplot(x=df[target_column])# like hist gram
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, 'Delivery_Time')

#imbalanced data

In [ ]:
# Task 1: Write your code here:Split the dataset into features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)# drop the Target
y = df['Delivery_Time']#save target here

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np
model = LogisticRegression()


print(f"After dropping missing price/year/odometer: {df_clean.shape}")
#Use the correct split: KFold OR StratifiedKFold
# StratifiedKFold (for imbalanced data)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(model, X, y, cv=skfold, scoring='accuracy')
print("\nStratifiedKFold scores:", scores_strat)
#print(f"Mean accuracy: {scores_strat.mean():.3f}")




In [ ]:
# Task 1: Write your code here:Plot feature importance from your trained model
# Retrieve CatBoost feature importances and sort them
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

catboost_model = models["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)
# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:Plot predicted delivery time histogram
# Plot Predictions vs Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Exam Scores (Ground Truth)")
plt.ylabel("Predicted Exam Scores")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: